# Macro-Driven Regime Switching Asset Allocation
### Dynamic Portfolio Timing between Equities (SPY) and Fixed Income (AGG)

This notebook provides an end-to-end, reproducible quantitative walk-through of our macro regime-switching asset allocation framework:
1. **Data Ingestion & Hygiene**: Fetches S&P 500 (SPY), Core Aggregate Bonds (AGG proxy-stitched with VBMFX pre-09/2003), Volatility (^VIX), and sovereign FRED macro indicators (Yield Curve Spread, Inflation, Fed Funds, Unemployment) with full Total Return adjustment (`auto_adjust=True`).
2. **Feature Engineering**: Constructs lagged economic momentum, inflation changes, realized volatility, and recession-stress metrics.
3. **Regime Switching Models**:
   * **Traditional**: 2-state Markov Regime-Switching Model (MRS) using the Hamilton Filter on market returns and volatility.
   * **Machine Learning**: Expanding-Window Walk-Forward Random Forest Classifier predicting forward downside distress without look-ahead bias.
4. **Backtesting & Execution Frictions**: Simulates both Binary (V1) and Continuous Blending (V2) allocations, deducting realistic 10 bps rebalancing fees and accounting for organic asset drift.
5. **Benchmark Comparison**: Evaluates strategies against the passive 60/40 SPY/AGG benchmark and pure S&P 500 Buy & Hold.

In [ ]:
# Automatically reload local source code updates
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load Data & Engineer Features

In [ ]:
from src.data_loader import prepare_dataset

# Prepares monthly data starting in 2000 (pre-2003 AGG history stitched with VBMFX)
df = prepare_dataset(start_date="2000-01-01")
print(f"Dataset shape: {df.shape}")
print(f"Available features: {list(df.columns)}")
df.tail()

## 2. Train Models & Generate Out-of-Sample Regimes

In [ ]:
from src.models import fit_markov_switching, run_expanding_window_ml

# Macro and market features for Random Forest
features_ml = [
    'SPY_3M_Ret', 'SPY_6M_Ret', 'SPY_12M_Ret',
    'AGG_3M_Ret', 'AGG_12M_Ret', 'SPY_Vol_3M',
    '10Y2Y_Yield_Spread', 'CPI_YoY', 'Fed_Funds_Rate',
    'Unemployment_Rate', 'Yield_Spread_Change_3M',
    'Fed_Funds_Change_3M', 'Unemployment_Change_3M'
]

# 1. Traditional Econometric Model: Markov Regime Switching
df['MRS_Prob'] = fit_markov_switching(df, features_for_mrs=['SPY_3M_Ret', 'SPY_Vol_3M'])

# 2. Machine Learning: Expanding-Window Walk-Forward Random Forest (120 months training startup)
df['ML_Prob'] = run_expanding_window_ml(df, features_ml, target_col='Regime_Target', min_train_months=120)

print("Probabilities generated successfully.")

## 3. Backtest Strategies vs. Passive Benchmarks

In [ ]:
from src.backtester import run_backtest, calculate_metrics, run_benchmark_60_40

# Baselines
spy_returns = df['SPY'].pct_change().fillna(0)
benchmark_returns = run_benchmark_60_40(df, risk_on_asset='SPY', risk_off_asset='AGG', transaction_cost=0.001)

# Version 1: Binary Allocation (100% SPY or 100% AGG at 50% threshold)
mrs_returns_v1, _ = run_backtest(df, df['MRS_Prob'], risk_on_asset='SPY', risk_off_asset='AGG', allocation_type='binary', threshold=0.5, transaction_cost=0.001)
ml_returns_v1, _ = run_backtest(df, df['ML_Prob'], risk_on_asset='SPY', risk_off_asset='AGG', allocation_type='binary', threshold=0.5, transaction_cost=0.001)

# Version 2: Continuous Allocation (Linear blending between SPY and AGG)
mrs_returns_v2, _ = run_backtest(df, df['MRS_Prob'], risk_on_asset='SPY', risk_off_asset='AGG', allocation_type='continuous', transaction_cost=0.001)
ml_returns_v2, _ = run_backtest(df, df['ML_Prob'], risk_on_asset='SPY', risk_off_asset='AGG', allocation_type='continuous', transaction_cost=0.001)

returns_all = pd.DataFrame({
    'S&P 500 Buy & Hold': spy_returns,
    'Benchmark (60/40)': benchmark_returns,
    'MRS - V1 (Binary)': mrs_returns_v1,
    'MRS - V2 (Continuous)': mrs_returns_v2,
    'ML (RF) - V1 (Binary)': ml_returns_v1,
    'ML (RF) - V2 (Continuous)': ml_returns_v2
}, index=df.index)

# Evaluation Window (Out-of-sample: 2010 to Present)
eval_start_date = df.index[120]
eval_returns = returns_all[returns_all.index >= eval_start_date]
eval_df = df[df.index >= eval_start_date]
print(f"Evaluation Window: {eval_start_date.strftime('%Y-%m')} to {eval_df.index[-1].strftime('%Y-%m')}")

## 4. Performance Metrics Comparison Table

In [ ]:
metrics_list = []
for col in eval_returns.columns:
    metrics = calculate_metrics(eval_returns[col])
    metrics['Strategy'] = col
    metrics_list.append(metrics)

metrics_df = pd.DataFrame(metrics_list).set_index('Strategy')
formatted = metrics_df.copy()
formatted['Cumulative Return'] = (formatted['Cumulative Return'] * 100).round(2).astype(str) + '%'
formatted['Annualized Return'] = (formatted['Annualized Return'] * 100).round(2).astype(str) + '%'
formatted['Annualized Vol'] = (formatted['Annualized Vol'] * 100).round(2).astype(str) + '%'
formatted['Sharpe Ratio'] = formatted['Sharpe Ratio'].round(3)
formatted['Max Drawdown'] = (formatted['Max Drawdown'] * 100).round(2).astype(str) + '%'
formatted

## 5. Visualizations: Cumulative Returns & Drawdowns

In [ ]:
from src.visualizer import plot_performance, plot_drawdowns, plot_combined_regimes
%matplotlib inline

# 1. Cumulative Wealth Curves
cum_returns = (1 + eval_returns).cumprod() - 1
fig, ax = plt.subplots(figsize=(12, 6))
for col in ['S&P 500 Buy & Hold', 'Benchmark (60/40)', 'ML (RF) - V1 (Binary)', 'ML (RF) - V2 (Continuous)']:
    ax.plot(cum_returns.index, cum_returns[col] * 100, label=col, linewidth=2)
ax.set_title("Dynamic Asset Allocation (AGG) - Cumulative Returns", fontsize=14, fontweight='bold')
ax.set_ylabel("Cumulative Return (%)")
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.show()

# 2. Combined Regime Probabilities with Colored Shading
plot_combined_regimes(eval_df['SPY'], eval_df['MRS_Prob'], eval_df['ML_Prob'], 
                      title="S&P 500 & Regime Probabilities (Blue=MRS, Green=ML)", 
                      save_path="../plots/regime_switching_combined.png")